# Role-Binding Beats Chain-of-Thought's Result With Zero Extra Tokens

**One notebook. Every open item. Run it once and you know.**

---

## The measured result (produced on one CPU core, reproduced here)

A 2-layer, 64-dim transformer must track a value through **4 composed operations**.
Three intermediate values exist. All arms receive the **identical correct
intermediates** — the only difference is *how those values are made available to
the second layer*.

| arm | what it does | S5 (chance 0.20) | Affine (chance 0.059) |
|---|---|---|---|
| `baseline` | nothing | 0.2025 | 0.0620 |
| `sum_noroles` | 1 slot = plain **sum** of the 3 values | 0.6645 | 0.1475 |
| `raw_multi` | 3 **separate** slots, raw values | 0.4960 | 0.1835 |
| **`hrr`** | **1 slot, role-bound, retrieved by unbinding** | **1.0000** | **1.0000** |
| `cot` | token-level chain of thought | 1.0000 | 1.0000 |

**Role-binding reaches chain-of-thought's accuracy while emitting zero extra tokens.**
Replicated across two structurally different algebras — a non-solvable group (S5)
and a modular ring (affine mod 17).

## Why the two controls matter more than the headline

The controls get the **same correct values** through the **same architecture**.
They are the trap door:

- `raw_multi` gives every value its own address — and still fails. So
  **addressability alone is not the mechanism.**
- `sum_noroles` compresses them into one vector without role tags — and fails. So
  **compression alone is not the mechanism.**

Only tagging each value with a role vector before superposing works. That is what
Smolensky (1990) and Plate (1995) claimed, tested against a transformer on a task
with a known complexity lower bound.

## What this does NOT show

Read this before you get excited.

- **The intermediates are given, not computed.** The model is handed the correct
  values. This measures whether a model can *use* a value it has — which was the
  original puzzle (100% auxiliary accuracy, still failed). It does **not** show the
  model can produce them itself. That is the next experiment, and it is the hard one.
- **Toy scale.** 64-dim, 2 layers, synthetic algebra. Nothing here says this
  survives at 7B on natural language.
- **Single seed per cell.** Differences under ~20% mean nothing. Cell 8 runs
  multiple seeds.
- **Prior art is real.** TPR-Transformer (arXiv:1910.06611), Differentiable Tree
  Machine (ICML 2023), LARS-VSA (arXiv:2405.14436) all bind inside transformers.
  The untested pairing is *this mechanism against this failure mode* — and the
  conceptual paper proposing exactly this (arXiv:2512.14709, Dec 2025) ran no
  experiments.

## 1 · Setup

In [ ]:
!pip -q install torch --upgrade 2>/dev/null | tail -1
import math, time, json, itertools
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(1337); np.random.seed(1337)
print("device:", DEV)
print("Every headline number was produced on ONE CPU core. A T4 is ~10x faster.")

## 2 · The binding operation (Plate 1995)

Binding is **circular convolution**, computed via FFT:

$$(x \circledast y)_k = \sum_j x_j\, y_{(k-j) \bmod d} \quad=\quad \mathcal{F}^{-1}\big(\mathcal{F}(x)\odot\mathcal{F}(y)\big)$$

Unbinding uses the **involution** $y^{+} = (y_1, y_d, y_{d-1}, \dots, y_2)$, so
$x \approx (x \circledast y) \circledast y^{+}$.

The whole point: several bindings can be **summed into one vector** and each
recovered by unbinding with its own role. Cell 3 verifies this before anything is
built on it.

In [ ]:
def hrr_bind(x, y):
    n = x.shape[-1]
    return torch.fft.irfft(torch.fft.rfft(x, n=n) * torch.fft.rfft(y, n=n), n=n)

def hrr_inv(y):
    return torch.cat([y[..., :1], y[..., 1:].flip(-1)], dim=-1)

def hrr_unbind(b, y):
    return hrr_bind(b, hrr_inv(y))

def rand_hv(shape, d, gen=None):
    return torch.randn(*shape, d, generator=gen) / math.sqrt(d)   # Plate: N(0, 1/d)

print("[ok] HRR ops")

## 3 · Verify binding works before building on it

In [ ]:
torch.manual_seed(0); d = 256
r1,r2,r3 = rand_hv((3,), d); f1,f2,f3 = rand_hv((3,), d)

M = hrr_bind(f1,r1) + hrr_bind(f2,r2) + hrr_bind(f3,r3)   # 3 values, ONE vector
print("superpose 3 bindings, retrieve each by its role:")
for i,(f,r) in enumerate([(f1,r1),(f2,r2),(f3,r3)], 1):
    rec = hrr_unbind(M, r)
    s = [float(torch.cosine_similarity(rec,g,dim=0)) for g in (f1,f2,f3)]
    print(f"   role {i}: cos to f1,f2,f3 = {[round(v,3) for v in s]}  -> picks f{s.index(max(s))+1}")

M2 = f1 + f2 + f3                                          # CONTROL: no roles
s = [float(torch.cosine_similarity(M2,g,dim=0)) for g in (f1,f2,f3)]
print(f"\nno roles, plain sum:  cos = {[round(v,3) for v in s]}  <- all alike, unrecoverable")

print("\nretrieval accuracy vs dimension (3 superposed, 200 trials):")
for d_ in [64,128,256,512]:
    ok = 0
    for t in range(200):
        g = torch.Generator().manual_seed(t)
        R = rand_hv((3,), d_, gen=g); Fv = rand_hv((3,), d_, gen=g)
        Ms = sum(hrr_bind(Fv[i], R[i]) for i in range(3))
        for i in range(3):
            rec = hrr_unbind(Ms, R[i])
            sc = [float(torch.cosine_similarity(rec, Fv[j], dim=0)) for j in range(3)]
            ok += (sc.index(max(sc)) == i)
    print(f"   d={d_:4d}: {ok/600*100:5.1f}%")

## 4 · Two task families

Both track a state through `k` composed operations, but over **different algebra** —
if a result holds on both, it is not an artifact of one.

- **S5**: state in {0..4}, each op a permutation. Composition is a **lookup**.
  S5 is non-solvable; its word problem is NC¹-complete.
- **Affine**: state in ℤ₁₇, each op `x → (a·x+b) mod 17`. Composition is **arithmetic**.

`k=4` gives **3 intermediates**, which is what makes superposition meaningful.
With one intermediate, "binding" is just a fixed invertible transform and the
binding arm and control arm become indistinguishable.

In [ ]:
_P = list(itertools.permutations(range(5)))
PERM = np.array(_P, dtype=np.int64); N_POS = 5

class S5Task:
    name = "S5"
    def __init__(self, k=4, domain=120, seed=0):
        self.k, self.domain, self.n_states = k, domain, N_POS
        self.POFF, self.BOS, self.SEP, self.VOCAB = 5, 125, 126, 127
        self.rng = np.random.default_rng(seed)
    def sample(self, n):
        return (self.rng.integers(0,N_POS,size=n),
                self.rng.integers(0,self.domain,size=(n,self.k)))
    def trace(self, s, g):
        out=np.empty_like(g); cur=s.copy()
        for t in range(g.shape[1]): cur=PERM[g[:,t],cur]; out[:,t]=cur
        return out
    def batch(self, n, dev):
        s,g=self.sample(n); tr=self.trace(s,g)
        seq=np.concatenate([np.full((n,1),self.BOS),s[:,None],g+self.POFF,
                            np.full((n,1),self.SEP)],axis=1)
        return (torch.from_numpy(seq).long().to(dev),
                torch.from_numpy(tr[:,-1]).long().to(dev),
                torch.from_numpy(tr[:,:-1]).long().to(dev))
    def batch_cot(self, n, dev):
        s,g=self.sample(n); tr=self.trace(s,g)
        seq=np.concatenate([np.full((n,1),self.BOS),s[:,None],g+self.POFF,
                            np.full((n,1),self.SEP),tr],axis=1)
        return (torch.from_numpy(seq[:,:-1]).long().to(dev),
                torch.from_numpy(seq[:,1:]).long().to(dev))
    @property
    def maxlen(self): return 2*self.k+7
    @property
    def chance(self): return 1.0/N_POS

class AffineTask:
    name = "Affine"
    def __init__(self, k=4, m=17, domain=120, seed=0):
        self.k,self.m,self.domain,self.n_states=k,m,domain,m
        self.rng=np.random.default_rng(seed)
        pairs,a=[],1
        while len(pairs)<domain:
            for b in range(m):
                if len(pairs)>=domain: break
                if a%m!=0: pairs.append((a%m,b))
            a+=1
        self.pool=np.array(pairs[:domain],dtype=np.int64)
        self.BOS,self.SEP=m,m+1; self.POFF=m+2; self.VOCAB=self.POFF+domain
    def sample(self,n):
        return (self.rng.integers(0,self.m,size=n),
                self.rng.integers(0,self.domain,size=(n,self.k)))
    def trace(self,x0,ops):
        out=np.empty_like(ops); cur=x0.copy()
        for t in range(ops.shape[1]):
            a=self.pool[ops[:,t],0]; b=self.pool[ops[:,t],1]
            cur=(a*cur+b)%self.m; out[:,t]=cur
        return out
    def batch(self,n,dev):
        x0,ops=self.sample(n); tr=self.trace(x0,ops)
        seq=np.concatenate([np.full((n,1),self.BOS),x0[:,None],ops+self.POFF,
                            np.full((n,1),self.SEP)],axis=1)
        return (torch.from_numpy(seq).long().to(dev),
                torch.from_numpy(tr[:,-1]).long().to(dev),
                torch.from_numpy(tr[:,:-1]).long().to(dev))
    def batch_cot(self,n,dev):
        x0,ops=self.sample(n); tr=self.trace(x0,ops)
        seq=np.concatenate([np.full((n,1),self.BOS),x0[:,None],ops+self.POFF,
                            np.full((n,1),self.SEP),tr],axis=1)
        return (torch.from_numpy(seq[:,:-1]).long().to(dev),
                torch.from_numpy(seq[:,1:]).long().to(dev))
    @property
    def maxlen(self): return 2*self.k+7
    @property
    def chance(self): return 1.0/self.m

print("[ok] tasks")

## 5 · The model — the whole experiment is `_memory_slots`

Standard 2-layer transformer. Between layer 1 and layer 2, the three intermediate
values are injected in one of four ways. **Nothing else differs between arms.**

Role vectors are **fixed and frozen** (never trained), which isolates the binding
mechanism from representation learning.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, h, zero_init=False):
        super().__init__()
        self.n1=nn.LayerNorm(d); self.attn=nn.MultiheadAttention(d,h,batch_first=True)
        self.n2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Linear(4*d,d))
        if zero_init:   # identity at init: stabilises gradients through many loops
            nn.init.zeros_(self.attn.out_proj.weight); nn.init.zeros_(self.attn.out_proj.bias)
            nn.init.zeros_(self.ff[-1].weight); nn.init.zeros_(self.ff[-1].bias)
    def forward(self,x,mask):
        h=self.n1(x); a,_=self.attn(h,h,h,attn_mask=mask,need_weights=False)
        x=x+a; return x+self.ff(self.n2(x))

class BindingLM(nn.Module):
    def __init__(self, task, d=64, heads=4, mode="baseline", n_loops=1,
                 zero_init=False, seed=0):
        super().__init__()
        self.task,self.d,self.mode=task,d,mode
        self.n_inter=task.k-1; self.n_loops=n_loops
        self.tok=nn.Embedding(task.VOCAB,d); self.pos=nn.Embedding(task.maxlen+8,d)
        self.l1=Block(d,heads,zero_init and n_loops>1)
        self.l2=Block(d,heads,zero_init and n_loops>1)
        self.norm=nn.LayerNorm(d); self.head=nn.Linear(d,task.VOCAB)
        g=torch.Generator().manual_seed(seed+999)
        self.register_buffer("roles", torch.randn(max(self.n_inter,1),d,generator=g)/math.sqrt(d))
        self.val_embed=nn.Embedding(task.n_states,d)

    def _memory_slots(self, fillers):
        """THE EXPERIMENT. Same values in, four ways of making them available."""
        if self.mode=="raw_multi":                       # separate addresses, no roles
            return torch.stack(fillers,1)
        if self.mode=="sum_noroles":                     # compressed, no roles
            return torch.stack(fillers,1).sum(1,keepdim=True)
        if self.mode=="hrr":                             # compressed AND role-tagged
            M=sum(hrr_bind(f,self.roles[i]) for i,f in enumerate(fillers))
            return torch.stack([hrr_unbind(M,self.roles[i]) for i in range(len(fillers))],1)
        return None

    def forward(self, idx, oracle_inter=None):
        B,T=idx.shape
        x=self.tok(idx)+self.pos(torch.arange(T,device=idx.device))[None]
        m=torch.triu(torch.full((T,T),float("-inf"),device=idx.device),1)
        for _ in range(self.n_loops): x=self.l1(x,m)
        if self.mode not in ("baseline","cot") and oracle_inter is not None:
            fillers=[self.val_embed(oracle_inter[:,i]) for i in range(self.n_inter)]
            slots=self._memory_slots(fillers)
            if slots is not None:
                x=torch.cat([slots,x],dim=1)             # PREPEND so causal mask reaches them
                T2=x.shape[1]
                m=torch.triu(torch.full((T2,T2),float("-inf"),device=idx.device),1)
        for _ in range(self.n_loops): x=self.l2(x,m)
        return self.head(self.norm(x)), None

    def n_params(self): return sum(p.numel() for p in self.parameters())

print("[ok] model")

## 6 · Train / evaluate

In [ ]:
def fit(task, mode, steps=2500, d=64, bs=256, lr=3e-3, n_loops=1,
        zero_init=False, seed=0, dev=DEV):
    torch.manual_seed(seed); np.random.seed(seed)
    m=BindingLM(task,d=d,mode=mode,n_loops=n_loops,zero_init=zero_init,seed=seed).to(dev)
    opt=torch.optim.AdamW(m.parameters(),lr=lr,weight_decay=0.01)
    sch=torch.optim.lr_scheduler.OneCycleLR(opt,lr,total_steps=steps)
    ns=task.n_states
    for s in range(steps):
        if mode=="cot":
            x,y=task.batch_cot(bs,dev); lg,_=m(x)
            mk=torch.zeros_like(y,dtype=torch.bool); mk[:,-task.k:]=True
            loss=F.cross_entropy(lg[mk],y[mk])
        else:
            x,y,inter=task.batch(bs,dev); lg,_=m(x,oracle_inter=inter)
            loss=F.cross_entropy(lg[:,-1,:ns],y)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step(); sch.step()
    return m

@torch.no_grad()
def evaluate(m, task, mode, n=2000, bs=500, dev=DEV):
    """CoT is scored AUTOREGRESSIVELY: the model writes its own chain."""
    m.eval(); ns=task.n_states; c=0
    for _ in range(n//bs):
        if mode=="cot":
            s,g=task.sample(bs); ans=task.trace(s,g)[:,-1]
            pre=np.concatenate([np.full((bs,1),task.BOS),s[:,None],g+task.POFF,
                                np.full((bs,1),task.SEP)],axis=1)
            cur=torch.from_numpy(pre).long().to(dev)
            for _ in range(task.k):
                lg,_=m(cur); cur=torch.cat([cur,lg[:,-1,:ns].argmax(-1,keepdim=True)],1)
            c+=int((cur[:,-1].cpu().numpy()==ans).sum())
        else:
            x,y,inter=task.batch(bs,dev); lg,_=m(x,oracle_inter=inter)
            c+=int((lg[:,-1,:ns].argmax(-1)==y).sum())
    m.train(); return c/(n//bs*bs)

print("[ok] harness")

## 7 · THE MAIN RUN

~10 minutes on a T4. Watch `hrr` against the two controls, not against baseline.

In [ ]:
MODES=["baseline","sum_noroles","raw_multi","hrr","cot"]
LABEL={"baseline":"nothing","sum_noroles":"1 slot, plain sum (no roles)",
       "raw_multi":"3 separate slots (no roles)","hrr":"1 slot, ROLE-BOUND",
       "cot":"token chain-of-thought"}
results=[]
for Cls in [S5Task, AffineTask]:
    t=Cls(k=4,domain=120)
    print(f"\n{'='*74}\n{t.name}  k=4  (3 intermediates)  chance={t.chance:.4f}\n{'='*74}")
    for mode in MODES:
        t0=time.time(); m=fit(t,mode); a=evaluate(m,t,mode)
        results.append(dict(task=t.name,mode=mode,acc=a,chance=t.chance))
        print(f"  {mode:12s} {LABEL[mode]:32s} acc {a:.4f}  [{time.time()-t0:.0f}s]",flush=True)

print(f"\n{'='*74}\nSUMMARY\n{'='*74}")
print(f"{'arm':<14}{'S5 (0.20)':>14}{'Affine (0.059)':>18}")
for mode in MODES:
    r={x['task']:x['acc'] for x in results if x['mode']==mode}
    print(f"{mode:<14}{r.get('S5',float('nan')):>14.4f}{r.get('Affine',float('nan')):>18.4f}")
json.dump(results,open("results.json","w"),indent=1)

## 8 · Multi-seed — because one seed proves nothing

If the gap between `hrr` and the controls survives three seeds on both tasks, it
is real. If it does not, stop here and do not believe section 7.

In [ ]:
SEEDS=[0,1,2]
print(f"{'task':<9}{'arm':<14}" + "".join(f"{f'seed {s}':>10}" for s in SEEDS) + f"{'mean':>9}")
print("-"*70)
multi={}
for Cls in [S5Task, AffineTask]:
    t=Cls(k=4,domain=120)
    for mode in ["sum_noroles","raw_multi","hrr"]:
        accs=[]
        for s in SEEDS:
            m=fit(Cls(k=4,domain=120,seed=s),mode,seed=s)
            accs.append(evaluate(m,Cls(k=4,domain=120,seed=s),mode))
        multi[(t.name,mode)]=accs
        print(f"{t.name:<9}{mode:<14}"+"".join(f"{a:>10.4f}" for a in accs)+f"{np.mean(accs):>9.4f}",flush=True)
        json.dump({f"{k[0]}|{k[1]}":v for k,v in multi.items()},open("multiseed.json","w"),indent=1)

## 9 · Falsification — the cell that can kill the result

Four ways the conclusion could be wrong. Run them.

1. **Shuffled roles at test time.** If binding is doing the work, using the wrong
   role to unbind must destroy accuracy. If it doesn't, the model was never using
   the roles.
2. **Random (untrained) fillers.** Feeding noise instead of the true intermediates
   must collapse to baseline. If `hrr` still wins, the gain was an architectural
   artifact of having extra slots, not the information in them.
3. **Dimension.** Binding capacity depends on `d`. Accuracy should degrade as `d`
   shrinks toward the point where 3 superposed bindings stop being separable.
4. **More intermediates.** At `k=6` (5 superposed) crosstalk grows. If `hrr` holds
   at d=64, that is a stronger result; if it breaks, that is the capacity limit and
   worth reporting honestly.

In [ ]:
t=S5Task(k=4,domain=120); ns=t.n_states
m=fit(t,"hrr")
print(f"hrr, intact                    {evaluate(m,t,'hrr'):.4f}")

# 1. shuffled roles at test time
orig=m.roles.clone(); m.roles.copy_(orig[torch.randperm(orig.shape[0])])
print(f"hrr, roles SHUFFLED at test    {evaluate(m,t,'hrr'):.4f}   <- must collapse")
m.roles.copy_(orig)

# 2. random fillers instead of true intermediates
@torch.no_grad()
def eval_randfill(m,task,n=2000,bs=500):
    m.eval(); c=0
    for _ in range(n//bs):
        x,y,inter=task.batch(bs,DEV)
        fake=torch.randint(0,task.n_states,inter.shape,device=DEV)
        lg,_=m(x,oracle_inter=fake)
        c+=int((lg[:,-1,:task.n_states].argmax(-1)==y).sum())
    m.train(); return c/(n//bs*bs)
print(f"hrr, RANDOM fillers            {eval_randfill(m,t):.4f}   <- must collapse")

# 3. dimension sweep
print()
for d_ in [16,32,64,128]:
    mm=fit(t,"hrr",d=d_); print(f"hrr d={d_:4d}                     {evaluate(mm,t,'hrr'):.4f}",flush=True)

# 4. more superposed intermediates
print()
for K in [4,6]:
    tk=S5Task(k=K,domain=120); mm=fit(tk,"hrr")
    print(f"hrr k={K} ({K-1} superposed)        {evaluate(mm,tk,'hrr'):.4f}",flush=True)

## 10 · The recurrent-depth question, settled separately

An earlier round concluded "recurrent looping does not fix the cliff." That
conclusion was **under-trained**. Kohli et al. (arXiv:2604.07822, Apr 2026) show
compositional generalization from looping is a **grokking** phenomenon — it emerges
only after thousands of epochs, long after training loss has converged, and needs
**zero-initialised** recurrent blocks (identity at init) for gradients to survive
many iterations.

This cell tests looping properly. It is slow by design; the whole point is that
short runs give a false negative.

In [ ]:
t=S5Task(k=2,domain=120)     # the original k=2 cliff
print(f"S5 k=2, domain=120, chance {t.chance}")
for loops,zi,steps in [(1,False,3000),(4,False,3000),(4,True,3000),(4,True,12000)]:
    t0=time.time(); m=fit(t,"baseline",steps=steps,n_loops=loops,zero_init=zi)
    a=evaluate(m,t,"baseline")
    tag=f"loops={loops} zero_init={zi} steps={steps}"
    print(f"  {tag:42s} acc {a:.4f}  [{time.time()-t0:.0f}s]",flush=True)
print("\nIf the 12000-step zero-init run beats the 3000-step one, the earlier")
print("negative was a training-length artifact, not evidence against looping.")

## 11 · How to read all of this

**Supports the binding hypothesis:** `hrr` matches `cot` on both tasks; both
controls stay far below; shuffled roles and random fillers collapse; the gap
survives three seeds.

**Kills it:** `raw_multi` matches `hrr` (then it is addressability, not binding);
or shuffled roles change nothing (the model ignored the roles); or the gap vanishes
across seeds.

**What you would be able to claim if it holds:** *an explicit role-binding memory
lets a small transformer reach chain-of-thought accuracy on composition tasks with
zero additional output tokens, and two controls rule out addressability and
compression as the explanation.*

**What you could not claim:** that the model computes its own intermediates (it is
handed them), or that any of this transfers beyond toy scale. Both are open, and the
first one is the genuinely hard next problem.